# 📦 PackVision AI — Free Google Colab GPU Worker (T4 16GB)
This notebook turns a free Google Colab instance into a high-performance **Stable Diffusion 1.5 + ControlNet Canny** GPU worker for your local PackVision AI Studio!

### Step-by-Step Instructions:
1. **Switch to GPU**: Click **Runtime** → **Change runtime type** → select **T4 GPU** → click **Save**.
2. **Run Cell 1**: Installs PyTorch, Hugging Face Diffusers, ControlNet, FastAPI, and Cloudflared.
3. **Run Cell 2**: Loads the AI models and outputs a **Public URL** (e.g. `https://xxxx.trycloudflare.com`).
4. **Connect Local Studio**: Open your local PackVision Studio → **Settings & Hardware** → paste the URL into **Google Colab URL** and click **Test / Connect**.
5. Return to **Studio Workbench** and hit **Generate Packaging Render** — your local studio will now render using this 16GB T4 GPU!

In [ ]:
# Step 1: Install Dependencies
!pip install -q diffusers transformers accelerate torch torchvision fastapi uvicorn pydantic pillow
# Download cloudflared tunnel binary (free, no sign-up or auth token needed!)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("Dependencies installed successfully!")

In [ ]:
# Step 2: Start Colab AI Worker Server & Cloudflare Public Tunnel
import os, io, gc, time, base64, subprocess, threading, re
from typing import Optional
import torch
from PIL import Image
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
import uvicorn

app = FastAPI(title="PackVision Colab GPU Worker")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device Check] Using {device.upper()}: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

print("[Models] Downloading and loading SD 1.5 + ControlNet Canny in FP16...")
cnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_canny", torch_dtype=torch.float16 if device == "cuda" else torch.float32)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=cnet,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)
if device == "cuda":
    pipe.enable_attention_slicing()
print("[Models] Pipeline ready for inference!")

class RenderPayload(BaseModel):
    image: str
    prompt: str
    negative_prompt: Optional[str] = "deformed, blurry, bad anatomy, bad art, distorted edges"
    steps: int = 30
    guidance_scale: float = 7.5
    controlnet_scale: float = 0.85
    seed: Optional[int] = None
    width: int = 512
    height: int = 512

@app.get("/health")
def health():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram = int(torch.cuda.get_device_properties(0).total_memory / (1024*1024)) if torch.cuda.is_available() else 0
    return {"status": "online", "service": "PackVision Colab Worker", "gpu": gpu, "vram_mb": vram, "models_loaded": True}

@app.post("/render")
def render(payload: RenderPayload):
    t0 = time.time()
    cond_img = Image.open(io.BytesIO(base64.b64decode(payload.image))).convert("RGB").resize((payload.width, payload.height))
    generator = torch.Generator(device="cpu").manual_seed(payload.seed) if payload.seed is not None else None
    out = pipe(
        prompt=payload.prompt,
        negative_prompt=payload.negative_prompt,
        image=cond_img,
        num_inference_steps=payload.steps,
        guidance_scale=payload.guidance_scale,
        controlnet_conditioning_scale=payload.controlnet_scale,
        generator=generator,
    )
    buf = io.BytesIO()
    out.images[0].save(buf, format="PNG")
    b64_res = base64.b64encode(buf.getvalue()).decode("utf-8")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    return {"success": True, "image": b64_res, "inference_time_s": round(time.time() - t0, 2), "device": device}

# Start FastAPI in background thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")
threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# Start Cloudflare Tunnel to get a secure public HTTPS URL
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*60)
print(f"🔥 PACKVISION COLAB GPU WORKER IS LIVE!")
print(f"👉 PUBLIC URL: {public_url}")
print("="*60)
print("Copy the PUBLIC URL above and paste it into:")
print("PackVision Studio -> Settings & Hardware -> Google Colab URL")
print("="*60 + "\n")

# Keep cell alive to log requests
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    tunnel_proc.terminate()
